#injesting the data in

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
from pyspark.sql.functions import current_timestamp
import yaml

#reading the data in df

In [0]:

with open("../config/pipeline_config.yaml") as pc:
  config = yaml.safe_load(pc)


accounts_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_ref", StringType(), False),
    StructField("account_type", StringType(), False),
    StructField("account_status", StringType(), False),
    StructField("open_date", StringType(), False),
    StructField("product_tier", StringType(), False),
    StructField("mobile_number", StringType(), True),
    StructField("digital_channel", StringType(), False),
    StructField("credit_limit", DoubleType(), True),
    StructField("current_balance", DoubleType(), False),
    StructField("last_activity_date", StringType(), True)])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("id_number", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("dob", StringType(), False),
    StructField("gender", StringType(), False),
    StructField("province", StringType(), False),
    StructField("income_band", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("risk_score", IntegerType(), False),
    StructField("kyc_status", StringType(), False),
    StructField("product_flags", StringType(), False)
  ])

trasactions_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("transaction_date", StringType(), False),
    StructField("transaction_time", StringType(), False),
    StructField("transaction_type", StringType(), False),
    StructField("merchant_category", StringType(), True),
    StructField("merchant_subcategory", StringType(), True),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False)
    StructField("channel", StringType(), False),
    StructField("location.province", StringType(), True),
    StructField("location.city", StringType(), True),
    StructField("location.coordinates", StringType(), True),
    StructField("metadata.device_id", StringType(), True),
    StructField("metadata.session_id", StringType(), True),
    StructField("metadata.retry_flag", BooleanType(), True),
  ])  

#accounts 
df = spark.read.csv(config["input"]["accounts_path"], header=True, schema=accounts_schema)
df = df.withColumn("ingestion_time", current_timestamp())
df.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .save(config["output"][bronze_path]+"/accounts")

#customers 
df = spark.read.csv(config["input"]["customers_path"], header=True, schema=customers_schema)
df = df.withColumn("ingestion_time", current_timestamp())
df.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .save(config["output"][bronze_path]+"/customers")

#transactions 
df = spark.read.csv(config["input"]["transactions_path"], header=True, schema=transactions_schema)
df = df.withColumn("ingestion_time", current_timestamp())
df.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .save(config["output"][bronze_path]+"/transactions")


#Customers load

In [0]:



schema_account = StructType([
    StructField("customer_id", StringType(), False),
    StructField("id_number", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("dob", StringType(), False),
    StructField("gender", StringType(), False),
    StructField("province", StringType(), False),
    StructField("income_band", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("risk_score", IntegerType(), False),
    StructField("kyc_status", StringType(), False),
    StructField("product_flags", StringType(), False)
    ])

df = spark.read.csv("/Volumes/nedbank_ovation/bronze/raw_data/customers.csv", header=True, schema=schema_account)

df = df.withColumn("ingestion_time", current_timestamp())

(
  df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nedbank_ovation.bronze.customers")
)

df.show()

In [0]:
%sql
SELECT * FROM nedbank_ovation.bronze.customers